In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
import json


In [2]:
# Generate a synthetic dataset with 50 input features
# Let's create 1000 samples with 50 features each and binary labels
np.random.seed(42)  # For reproducibility

X = np.random.rand(1000, 50)  # 1000 samples, 50 features
y = np.random.randint(2, size=(1000, 1))  # Binary labels (0 or 1)

print(X.shape, y.shape)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# print(y_train.shape, y_test.shape)
# print(X_train.shape, X_test.shape)


(1000, 50) (1000, 1)


In [3]:
# Define a single-layer perceptron model
model = Sequential([
    Dense(1, input_dim=50, activation='relu'),  # One neuron, 50 inputs, sigmoid activation
    # BatchNormalization()
])

# Compile the model
model.compile(optimizer='sgd',  # Stochastic Gradient Descent
              loss='binary_crossentropy',  # Loss function for binary classification
              metrics=['accuracy'])


# Train the model
model.fit(X_train, y_train, epochs=20, batch_size=10, verbose=1)


/home/nestor/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20


I0000 00:00:1730490409.840183   25043 service.cc:145] XLA service 0x7f522c004b70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1730490409.840204   25043 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 795us/step - accuracy: 0.4952 - loss: 7.2247
Epoch 2/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 249us/step - accuracy: 0.4948 - loss: 8.1423
Epoch 3/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 265us/step - accuracy: 0.4847 - loss: 8.3064
Epoch 4/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 324us/step - accuracy: 0.4905 - loss: 8.2126
Epoch 5/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 292us/step - accuracy: 0.4606 - loss: 8.6942
Epoch 6/20


I0000 00:00:1730490410.066679   25043 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 401us/step - accuracy: 0.4472 - loss: 8.9095
Epoch 7/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 275us/step - accuracy: 0.4690 - loss: 8.5586
Epoch 8/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 270us/step - accuracy: 0.4794 - loss: 8.3916
Epoch 9/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 266us/step - accuracy: 0.4794 - loss: 8.3906
Epoch 10/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 294us/step - accuracy: 0.5102 - loss: 7.8949
Epoch 11/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 301us/step - accuracy: 0.4859 - loss: 8.2868
Epoch 12/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 470us/step - accuracy: 0.5055 - loss: 7.9701
Epoch 13/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 425us/step - accuracy: 0.5030 - loss: 8.0106
Epoch 14/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 471us/step - accuracy: 0.4806 - loss: 8.3710
Epoch 15/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 284us/step - accuracy: 0.4881 - loss: 8.2504
Epoch 16/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 367us/step - accuracy: 0.4825 - loss: 8.3413
Epoch 17/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 322us/step - acc

In [4]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5340 - loss: 7.5109
Test Loss: 7.8979
Test Accuracy: 0.5100


In [5]:
# Access the weights
weights, bias = model.get_weights()

# Save the weights to a JSON file
weights_dict = {
    "weights": weights.tolist(),
    "bias": bias.tolist()
}

# print(weights)

In [6]:
with open("perceptron_weights.json", "w") as fw:
    json.dump(weights_dict, fw)

print("Weights and bias saved to perceptron_weights.json")

Weights and bias saved to perceptron_weights.json


In [7]:
# Read the JSON file
with open('perceptron_weights.json', 'r') as fr:
    data = json.load(fr)

# Convert the JSON data to a format suitable for Verilog
with open('weights_values.mem', 'w') as fmem:
    for weight_value in data['weights']:
        weight_valueQ15 = weight_value[0] * 2**15 # Convert to Q15.0 fixed-point format
        weight_valueQ15 = int(weight_valueQ15) # Convert to integer
        # Get the raw binary representation
        binary_representation = bin(weight_valueQ15 & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")
        # print(weight_valueQ15)    
        
    bias_value = data['bias']
    bias_valueQ15 = bias_value[0] * 2**15 # Convert to Q15.0 fixed-point format
    bias_valueQ15 = int(bias_valueQ15) # Convert to integer
    # Get the raw binary representation
    binary_representation = bin(bias_valueQ15 & 0xFFFF)[2:].zfill(16)
    fmem.write(f"{binary_representation}\n") 
    

# Save iput data to a file
XQ15 = X * 2**15
XQ15 = XQ15.astype(np.int16)

with open('input_values.mem', 'w') as fmem:
    for value in XQ15[0]:
        binary_representation = bin(value & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")  # Convert value to float before formatting as binary
# print(XQ15)

In [8]:
# Example single input (make sure it has the correct shape)
single_input = X[0].reshape(1, -1)

# Print the input value
# print("Input value for the single input:", single_input)

# Make a prediction
prediction = model.predict(single_input)

predictionQ15 = prediction[0][0] * 2**15 # Convert to Q15.0 fixed-point format

# Print the prediction and the classified class
print("Prediction for the single input:", prediction)
print("Prediction for the single input in Q15.0 format:", int(predictionQ15))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Prediction for the single input: [[0.]]
Prediction for the single input in Q15.0 format: 0


In [9]:
# Assuming single_input and weights are already defined as numpy arrays
# Slice the first ten elements
single_input_ = single_input[0]
weights_ = weights[0:50]

print(single_input_)
print(weights_)

# Perform the dot product with the first ten elements
product = np.dot(single_input_, weights_)
print("Product:", product)
print(product +bias_value[0])

[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864 0.15599452
 0.05808361 0.86617615 0.60111501 0.70807258 0.02058449 0.96990985
 0.83244264 0.21233911 0.18182497 0.18340451 0.30424224 0.52475643
 0.43194502 0.29122914 0.61185289 0.13949386 0.29214465 0.36636184
 0.45606998 0.78517596 0.19967378 0.51423444 0.59241457 0.04645041
 0.60754485 0.17052412 0.06505159 0.94888554 0.96563203 0.80839735
 0.30461377 0.09767211 0.68423303 0.44015249 0.12203823 0.49517691
 0.03438852 0.9093204  0.25877998 0.66252228 0.31171108 0.52006802
 0.54671028 0.18485446]
[[-0.36430112]
 [-0.7515273 ]
 [-0.110937  ]
 [-0.01518801]
 [-0.10510626]
 [-0.09441444]
 [-0.2574464 ]
 [-0.03701422]
 [ 0.19804561]
 [-0.498311  ]
 [ 0.09763724]
 [-0.00832042]
 [-0.08182561]
 [ 0.04720318]
 [-0.43921375]
 [-0.2792167 ]
 [-0.5148462 ]
 [-0.3536505 ]
 [ 0.01710729]
 [-0.5857754 ]
 [-0.16815218]
 [-0.11488201]
 [ 0.07622038]
 [-0.21622074]
 [-0.23330176]
 [-0.14331788]
 [ 0.03198822]
 [-0.35499373]
 [ 0.08695539]
 [-0.